# Kombinierte Labels: RBF-SVM ohne Metadaten

**Hypothese:** AEZ und Month liefern eigenständige Kontextinformation. Entfernt man sie, fällt die kombinierte BAcc gegenüber der SVM-Referenz mit Metadaten.

Die Modellfamilie, Skalierung und Hyperparametersuche bleiben unverändert. Nur `use_meta=False` unterscheidet diesen Lauf von der Referenz.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from awp2.experiment import RunConfig, TuneConfig, run, tune
from awp2.models import make_combined_svm
from awp2.preprocessing import PreprocessingConfig

COMBINED_SVM_CV_BACC = 0.834
COMBINED_SVM_VAL_BACC = 0.845

svm_no_meta_preprocessing = PreprocessingConfig(
    scale=True,
    use_meta=False,
)

## 1. Hyperparameter suchen

Die neun Kombinationen aus Regularisierung `C` und RBF-Bandbreite `gamma` entsprechen der SVM-Referenz. Sie werden ausschließlich anhand von `bacc_combined` auf den fünf gemeinsamen Trainings-Folds verglichen.

In [ ]:
tuned = tune(
    make_combined_svm(),
    TuneConfig(
        name="combined_svm_no_meta_search",
        approach="combined",
        description="RBF-SVM ohne AEZ und Month auf kombinierten Crop/Stage-Labels: C und gamma vergleichen.",
        preprocessing=svm_no_meta_preprocessing,
        param_grid={
            "estimator__C": [0.1, 1.0, 10.0],
            "estimator__gamma": ["scale", 0.001, 0.01],
        },
    ),
)

display(tuned.candidates.sort_values("rank"))
{
    "best_cv_score": tuned.best_score,
    "best_params": tuned.best_params,
    "tuning_run": tuned.run_id,
}

## 2. Einmal validieren

Nur der nach CV ausgewählte Kandidat wird auf dem gemeinsamen Validierungsanteil bewertet. Das Ergebnis prüft die Kontext-Hypothese auf einem Split, der weder zum Training noch zur Hyperparameterwahl verwendet wurde.

In [ ]:
result = run(
    tuned.best_model,
    RunConfig(
        name="combined_svm_no_meta",
        approach="combined",
        description=(
            "Beste RBF-SVM ohne AEZ und Month aus combined_svm_no_meta_search, "
            "einmal auf der Validierung bewertet."
        ),
        preprocessing=svm_no_meta_preprocessing,
        tuning_run=tuned.run_id,
    ),
)

result.metrics.model_dump()

## 3. Analyse

Die Vergleichstabelle zeigt den Kontextbeitrag getrennt für Trainings-CV und Validierung. Die Abbildungen verorten die Unterschiede in den Kandidaten, Klassenrecalls und häufigsten Verwechslungen.

In [ ]:
from awp2.data import combined_label
from awp2.evaluation import plot_confusion_matrices
from awp2.plots import plot_class_recall, plot_top_confusions, plot_tuning_candidates

comparison = pd.DataFrame(
    {
        "CV-BAcc kombiniert": {
            "SVM mit Metadaten": COMBINED_SVM_CV_BACC,
            "SVM ohne Metadaten": tuned.best_score,
        },
        "Validierungs-BAcc kombiniert": {
            "SVM mit Metadaten": COMBINED_SVM_VAL_BACC,
            "SVM ohne Metadaten": result.metrics.bacc_combined,
        },
    }
)
display(comparison)

top_confusions = (
    pd.DataFrame(
        {
            "Wahr": combined_label(result.y_val),
            "Vorhergesagt": combined_label(result.y_pred),
        }
    )
    .query("Wahr != Vorhergesagt")
    .value_counts()
    .head(10)
)
display(top_confusions)

tuning_ax = plot_tuning_candidates(
    tuned.candidates,
    color_by="estimator__C",
    validation_score=result.metrics.bacc_combined,
)
tuning_ax.set_title("Kombinierte Labels: CV-Score der SVM ohne Metadaten")
display(tuning_ax.figure)

for figure in (
    plot_confusion_matrices(result.y_val, result.y_pred),
    plot_class_recall(result.y_val, result.y_pred),
    plot_top_confusions(result.y_val, result.y_pred),
):
    display(figure)

In [ ]:
from awp2.config import MODEL_DOCS_FIGURES_DIR
from awp2.plots import save_doc_figure

save_doc_figure(
    tuning_ax.figure,
    "combined_svm_no_meta_tuning",
    MODEL_DOCS_FIGURES_DIR,
)

confusion_figure = plot_confusion_matrices(result.y_val, result.y_pred)
save_doc_figure(
    confusion_figure,
    "combined_svm_no_meta_confusion",
    MODEL_DOCS_FIGURES_DIR,
)

recall_figure = plot_class_recall(result.y_val, result.y_pred)
save_doc_figure(
    recall_figure,
    "combined_svm_no_meta_recall",
    MODEL_DOCS_FIGURES_DIR,
)

confusions_figure = plot_top_confusions(result.y_val, result.y_pred)
save_doc_figure(
    confusions_figure,
    "combined_svm_no_meta_confusions",
    MODEL_DOCS_FIGURES_DIR,
)

{
    "tuning_run": tuned.run_id,
    "evaluation_run": result.run_id,
    "invalid_combinations": result.metrics.invalid_combinations,
}